# Answer Key: U.S. Census Population Choropleth — Indiana

**Challenge:** Analyze U.S. Census population data for Indiana at the census-tract level.
1. Create a choropleth map of population across census tracts.
2. Compute the average tract population and produce a second map highlighting tracts above and below that average.

**Variable:** `DP02_0088E` — Total population (ACS 2023, 5-year estimates)  
**State:** Indiana (FIPS `18`)  
**Level:** Tracts

---

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# state fips code
fips = 18

## Step 1: Create and Read Base URL of Census Data with Pandas

In [ ]:
census_api_key = ""
baseurl = f'https://api.census.gov/data/2023/acs/acs5/profile?get=NAME,GEO_ID,DP02_0088E&for=tract:*&in=state:{fips}&in=county:*&descriptive=true&outputFormat=csv&key={census_api_key}'

In [ ]:
census = pd.read_csv(baseurl)
census = census[census['NAME'] != 'Geographic Area Name']
census

## Step 2: Clean the Census Data

In [ ]:
null_count_total = census.isnull().sum().sum()
print(f"Total number of NaN values in the DataFrame: {null_count_total}")

In [ ]:
census["GEOID"] = census["GEO_ID"].str.split("US").str[1]
census

In [ ]:
census["DP02_0088E"] = pd.to_numeric(census["DP02_0088E"], errors='coerce')

## Step 3: Download/Read and Merge the Shape File Data with Census

In [ ]:
url = f"https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_{fips:02d}_tract.zip"

In [ ]:
tract = gpd.read_file(url)

In [ ]:
tract

In [ ]:
# merge the datasets
census_tract = tract.merge(census, on="GEOID", how="left")
census_tract

## Step 4: Visualization - Maps

### 4A. Choropleth Map of Population

Color each census tract by its total population.  
Darker color = more people.

In [ ]:
population = "DP02_0088E"

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

census_tract.plot(
    column=population,
    cmap="YlOrRd",
    linewidth=0.15,
    edgecolor="grey",
    legend=True,
    ax=ax,
)

ax.set_title("Indiana Population 2023 ACS", fontsize=20)
ax.axis("off")
plt.tight_layout()
plt.show()

### 4B. Above vs Below Average Population

**Steps:**
1. Compute the mean (average) tract population.
2. Use a boolean mask to classify every tract as `"Above average"` or `"Below average"`.
3. Add that classification as a new column.
4. Map using a two-color categorical palette.

In [ ]:
# compute the average
avg = census_tract[population].mean()
print(f"Average tract population: {avg:,.1f}")

In [ ]:
# boolean mask
above_mask = census_tract[population] > avg

print(f"Tracts above average : {above_mask.sum()}")
print(f"Tracts below average : {(~above_mask).sum()}")

In [ ]:
# create classification column
census_tract["pop_class"] = above_mask.map({
    True:  "Above average",
    False: "Below average",
})

print(census_tract["pop_class"].value_counts())

In [ ]:
# plot the classification map
fig, ax = plt.subplots(figsize=(10, 10))

census_tract.plot(
    column="pop_class",
    categorical=True,
    cmap="YlGnBu_r",           # red = below, green = above
    linewidth=0.15,
    edgecolor="grey",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax,
)

ax.set_title(
    f"Indiana Census Tracts: Above vs. Below Average {population}\n"
    f"(Average = {avg:,.0f} people per tract, 2023 ACS)",
    fontsize=13,
)
ax.axis("off")
plt.tight_layout()
plt.show()